In [1]:
# Importar pacotes
import nltk
from nltk.classify import NaiveBayesClassifier
from nltk.corpus import subjectivity
from nltk.sentiment import SentimentAnalyzer
from nltk.sentiment.util import mark_negation, extract_unigram_feats

# Download subjectivity
nltk.download('subjectivity')

# Carregar docs
all_subj_docs = [(sent, 'subj') for sent in subjectivity.sents(categories='subj')]
all_obj_docs = [(sent, 'obj') for sent in subjectivity.sents(categories='obj')]

[nltk_data] Downloading package subjectivity to
[nltk_data]     C:\Users\franc\AppData\Roaming\nltk_data...
[nltk_data]   Package subjectivity is already up-to-date!


In [2]:
# Executar multiplos balancesmantos
def treinar_balancemantos(n_subj, n_obj):

    n_train_subj = int(n_subj*0.8)
    n_train_obj = int(n_obj*0.8)

    training_docs = all_subj_docs[:n_train_subj] + all_obj_docs[:n_train_obj]
    testing_docs  = all_subj_docs[n_train_subj:n_subj] + all_obj_docs[n_train_obj:n_obj]

    sentim_analyzer = SentimentAnalyzer()
    all_words_neg = sentim_analyzer.all_words([mark_negation(doc) for doc in training_docs])
    unigram_feats = sentim_analyzer.unigram_word_feats(all_words_neg, min_freq=4)
    sentim_analyzer.add_feat_extractor(extract_unigram_feats, unigrams=unigram_feats)

    sentim_analyzer.train(NaiveBayesClassifier.train, sentim_analyzer.apply_features(training_docs))
    return sentim_analyzer.evaluate(sentim_analyzer.apply_features(testing_docs))

In [3]:
#Executar treinamento para diferentes balancemantos
prop = [
    ("1:1", 500, 500),
    ("3:1", 750, 250),
    ("9:1", 900, 100)]

metrics = ["Accuracy",
           "F-measure [subj]", "F-measure [obj]",
           "Precision [subj]", "Precision [obj]",
           "Recall [subj]",    "Recall [obj]"]

resultados = {}
for cat, n_subj, n_obj in prop:
    resultados[cat] = treinar_balancemantos(n_subj, n_obj)

Training classifier
Evaluating NaiveBayesClassifier results...
Training classifier
Evaluating NaiveBayesClassifier results...
Training classifier
Evaluating NaiveBayesClassifier results...


In [4]:
#Exibir resultados
col_w = 20

header = f"{'Métrica':<22}" + "".join(f"{label:>{col_w}}" for label, *_ in prop)
print(header)
print("-" * 82)
for m in metrics:
    row = f"  {m:<20}"
    for label, *_ in prop:
        v = resultados[label].get(m) or 0
        row += f"{v:>{col_w}.4f}"
    print(row)
print("=" * 82)

Métrica                                1:1                 3:1                 9:1
----------------------------------------------------------------------------------
  Accuracy                          0.9050              0.9100              0.9000
  F-measure [subj]                  0.9045              0.9404              0.9454
  F-measure [obj]                   0.9055              0.8163              0.4118
  Precision [subj]                  0.9091              0.9342              0.9301
  Precision [obj]                   0.9010              0.8333              0.5000
  Recall [subj]                     0.9000              0.9467              0.9611
  Recall [obj]                      0.9100              0.8000              0.3500
